In [1]:
from huggingface_hub import login
import wandb
from sklearn.model_selection import train_test_split
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import (
    InformationRetrievalEvaluator,
    SequentialEvaluator,
)
from sentence_transformers import SentenceTransformerModelCardData
from sentence_transformers.util import cos_sim
import json
from datasets import load_dataset, concatenate_datasets
from sentence_transformers.losses import MatryoshkaLoss, MultipleNegativesRankingLoss

In [10]:
from datasets import load_dataset

corpus = load_dataset("another-symato/VMTEB-Zalo-legel-retrieval-wseg", "corpus", split="train")
queries = load_dataset("another-symato/VMTEB-Zalo-legel-retrieval-wseg", "queries", split="train")
relevant_docs_data = load_dataset("another-symato/VMTEB-Zalo-legel-retrieval-wseg", "data_ir", split="train")


2026-04-30 15:18:39 - HTTP Request: HEAD https://huggingface.co/datasets/another-symato/VMTEB-Zalo-legel-retrieval-wseg/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-30 15:18:39 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/another-symato/VMTEB-Zalo-legel-retrieval-wseg/a3c6685dd0e97ff94ef896f894ec23ce7f859e76/README.md "HTTP/1.1 200 OK"
2026-04-30 15:18:39 - HTTP Request: HEAD https://huggingface.co/datasets/another-symato/VMTEB-Zalo-legel-retrieval-wseg/resolve/a3c6685dd0e97ff94ef896f894ec23ce7f859e76/VMTEB-Zalo-legel-retrieval-wseg.py "HTTP/1.1 404 Not Found"
2026-04-30 15:18:39 - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/another-symato/VMTEB-Zalo-legel-retrieval-wseg/another-symato/VMTEB-Zalo-legel-retrieval-wseg.py "HTTP/1.1 404 Not Found"
2026-04-30 15:18:39 - HTTP Request: HEAD https://huggingface.co/datasets/another-symato/VMTEB-Zalo-legel-retrieval-wseg/resolve/a3c6685dd0e97ff94ef896f894ec2

In [20]:
type(corpus)

datasets.arrow_dataset.Dataset

In [21]:
df_corpus = corpus.to_pandas()
df_relevant_docs_data = relevant_docs_data.to_pandas()
df_queries = queries.to_pandas()
df_corpus.head()

,id,title,text
0,01/2009/tt-bnn#1,Điều 1. Phạm vi áp dụng,"Thông_tư này hướng_dẫn tuần_tra , canh_gác bảo..."
1,01/2009/tt-bnn#2,Điều 2. Tổ chức lực lượng,"1 . Hàng năm trước mùa mưa , lũ , Ủy_ban_nhân_..."
2,01/2009/tt-bnn#3,Điều 3. Tiêu chuẩn của các thành viên thuộc lự...,"1 . Là người khỏe mạnh , tháo_vát , đủ khả_năn..."
3,01/2009/tt-bnn#4,"Điều 4. Nhiệm vụ của lực lượng tuần tra, canh ...",1 . Chấp_hành sự phân_công của Ban chỉ_huy phò...
4,01/2009/tt-bnn#5,"Điều 5. Phù hiệu của lực lượng tuần tra, canh ...","Phù_hiệu của lực_lượng tuần_tra , canh_gác đê ..."


In [5]:
df_relevant_docs_data.head()


,corpus_id,query_id,score
0,11/2018/tt-byt#8,8f06f986ca91576bede33fdf5d5e1a2a,1
1,82/2020/nđ-cp#15,56a0c5f94a067bac466d3d72b74e781e,1
2,27/2019/nđ-cp#33,941db5fd20a02fe1311263d214d3c84e,1
3,214/2016/tt-btc#2,2975385f5935a17954dcab9ccc296b01,1
4,20/2020/nđ-cp#16,cbb9fc4580a226e586a1211f7b1fff06,1


In [11]:
df_queries.head()


,query_id,question
0,0637bf82c8b290c7875c5bfddbf91df5,Công_an xã xử_phạt lỗi không mang bằng lái_xe ...
1,ade2b2ee4f5b869f75f0d183902382af,Phải thực_hiện thao_tác_nạp mẫu vào bình_chứa ...
2,8fa164edc7a8419cd7dc9ce66d8e695a,Trình_tự đánh_giá chất_lượng hàng năm đối_với ...
3,3aed91309b72097b34427dd28bebd98a,Tử_tù chết sau khi tiêm thuốc_độc mà không có ...
4,fe9131a8862ce1bfa7de7e2b30eeb39e,Trách_nhiệm của Bộ Khoa_học_và Công_nghệ về qu...


In [22]:
print(len(queries))
print(len(corpus))

3196
61425


In [23]:
df_relevant_docs_data["score"].value_counts()

score
1    2556
Name: count, dtype: int64

# Model bi-endocder

In [ ]:
# Load a model
model_bi_encoder = SentenceTransformer("bkai-foundation-models/vietnamese-bi-encoder")


2026-04-28 14:17:06 - Use pytorch device_name: cuda:0
2026-04-28 14:17:06 - Load pretrained SentenceTransformer: bkai-foundation-models/vietnamese-bi-encoder
2026-04-28 14:17:06 - HTTP Request: HEAD https://huggingface.co/bkai-foundation-models/vietnamese-bi-encoder/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:17:06 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/bkai-foundation-models/vietnamese-bi-encoder/84f9d9ada0d1a3c37557398b9ae9fcedcdf40be0/modules.json "HTTP/1.1 200 OK"
2026-04-28 14:17:07 - HTTP Request: HEAD https://huggingface.co/bkai-foundation-models/vietnamese-bi-encoder/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:17:07 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/bkai-foundation-models/vietnamese-bi-encoder/84f9d9ada0d1a3c37557398b9ae9fcedcdf40be0/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-28 14:17:07 - HTTP Request: HEAD https://h

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-04-28 14:17:08 - HTTP Request: HEAD https://huggingface.co/bkai-foundation-models/vietnamese-bi-encoder/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:17:08 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/bkai-foundation-models/vietnamese-bi-encoder/84f9d9ada0d1a3c37557398b9ae9fcedcdf40be0/config.json "HTTP/1.1 200 OK"
2026-04-28 14:17:08 - HTTP Request: HEAD https://huggingface.co/bkai-foundation-models/vietnamese-bi-encoder/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:17:08 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/bkai-foundation-models/vietnamese-bi-encoder/84f9d9ada0d1a3c37557398b9ae9fcedcdf40be0/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-28 14:17:08 - HTTP Request: HEAD https://huggingface.co/bkai-foundation-models/vietnamese-bi-encoder/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:17:08 - HTTP Request: HEAD https://hugging

In [4]:
!pip install sentence_transformers

In [27]:
import logging
import random

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from datasets import load_dataset

logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)



# For this dataset, we want to concatenate the title and texts for the corpus
corpus = corpus.map(lambda x: {"text": (x["title"] + " " + x["text"]).strip()}, remove_columns=["title"])

# Shrink the corpus size heavily to only the relevant documents + 30,000 random documents
required_corpus_ids = set(map(str, relevant_docs_data["corpus_id"]))
required_corpus_ids |= set(random.sample(corpus["id"], k=30_000))
corpus = corpus.filter(lambda x: x["id"] in required_corpus_ids)

# Convert the datasets to dictionaries
corpus = dict(zip(corpus["id"], corpus["text"]))  # Our corpus (cid => document)
queries = dict(zip(queries["query_id"], queries["question"]))  # Our queries (qid => question)
relevant_docs = {}  # Query ID to relevant documents (qid => set([relevant_cids])
for qid, corpus_ids in zip(relevant_docs_data["query_id"], relevant_docs_data["corpus_id"]):
    qid = str(qid)
    corpus_ids = str(corpus_ids)
    if qid not in relevant_docs:
        relevant_docs[qid] = set()
    relevant_docs[qid].add(corpus_ids)

# Given queries, a corpus and a mapping with relevant documents, the InformationRetrievalEvaluator computes different IR metrics.
ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    mrr_at_k=[1, 3, 5, 10],
    ndcg_at_k=[1, 3, 5, 10],
    precision_recall_at_k=[1, 3, 5, 10],
    accuracy_at_k=[1,3,5,10],
    map_at_k=[10, 100],
    name="biencoder-subset-test",
)
results = ir_evaluator(model_bi_encoder)

Filter:   0%|          | 0/61425 [00:00<?, ? examples/s]

2026-04-28 14:17:45 - Information Retrieval Evaluation of the model on the biencoder-subset-test dataset:
2026-04-28 14:22:37 - Queries: 2556
2026-04-28 14:22:37 - Corpus: 30894

2026-04-28 14:22:37 - Score-Function: cosine
2026-04-28 14:22:37 - Accuracy@1: 69.99%
2026-04-28 14:22:37 - Accuracy@3: 84.82%
2026-04-28 14:22:37 - Accuracy@5: 88.89%
2026-04-28 14:22:37 - Accuracy@10: 92.14%
2026-04-28 14:22:37 - Precision@1: 69.99%
2026-04-28 14:22:37 - Precision@3: 28.27%
2026-04-28 14:22:37 - Precision@5: 17.78%
2026-04-28 14:22:37 - Precision@10: 9.21%
2026-04-28 14:22:37 - Recall@1: 69.99%
2026-04-28 14:22:37 - Recall@3: 84.82%
2026-04-28 14:22:37 - Recall@5: 88.89%
2026-04-28 14:22:37 - Recall@10: 92.14%
2026-04-28 14:22:37 - MRR@1: 0.6999
2026-04-28 14:22:37 - MRR@3: 0.7671
2026-04-28 14:22:37 - MRR@5: 0.7765
2026-04-28 14:22:37 - MRR@10: 0.7810
2026-04-28 14:22:37 - NDCG@1: 0.6999
2026-04-28 14:22:37 - NDCG@3: 0.7880
2026-04-28 14:22:37 - NDCG@5: 0.8049
2026-04-28 14:22:37 - NDCG@10:

In [ ]:
legal_dta_model = SentenceTransformer("MinhNguyet3001/embedding-supper-legal")


2026-04-28 14:23:27 - Use pytorch device_name: cuda:0
2026-04-28 14:23:27 - Load pretrained SentenceTransformer: tanh17042004/dta_legal_model_sup
2026-04-28 14:23:27 - HTTP Request: HEAD https://huggingface.co/tanh17042004/dta_legal_model_sup/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:23:28 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/tanh17042004/dta_legal_model_sup/00a2dbf1a1632387440b9898e2abff4a0aa6ec25/modules.json "HTTP/1.1 200 OK"
2026-04-28 14:23:28 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/tanh17042004/dta_legal_model_sup/00a2dbf1a1632387440b9898e2abff4a0aa6ec25/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

2026-04-28 14:23:28 - HTTP Request: HEAD https://huggingface.co/tanh17042004/dta_legal_model_sup/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:23:28 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/tanh17042004/dta_legal_model_sup/00a2dbf1a1632387440b9898e2abff4a0aa6ec25/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-28 14:23:28 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/tanh17042004/dta_legal_model_sup/00a2dbf1a1632387440b9898e2abff4a0aa6ec25/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

2026-04-28 14:23:28 - HTTP Request: HEAD https://huggingface.co/tanh17042004/dta_legal_model_sup/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:23:28 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/tanh17042004/dta_legal_model_sup/00a2dbf1a1632387440b9898e2abff4a0aa6ec25/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-28 14:23:28 - HTTP Request: HEAD https://huggingface.co/tanh17042004/dta_legal_model_sup/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:23:28 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/tanh17042004/dta_legal_model_sup/00a2dbf1a1632387440b9898e2abff4a0aa6ec25/README.md "HTTP/1.1 200 OK"
2026-04-28 14:23:28 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/tanh17042004/dta_legal_model_sup/00a2dbf1a1632387440b9898e2abff4a0aa6ec25/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

... (14 outputs left out)

bpe.codes: 0.00B [00:00, ?B/s]

2026-04-28 14:23:42 - HTTP Request: HEAD https://huggingface.co/tanh17042004/dta_legal_model_sup/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:23:42 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/tanh17042004/dta_legal_model_sup/00a2dbf1a1632387440b9898e2abff4a0aa6ec25/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-04-28 14:23:42 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/tanh17042004/dta_legal_model_sup/00a2dbf1a1632387440b9898e2abff4a0aa6ec25/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

2026-04-28 14:23:42 - HTTP Request: GET https://huggingface.co/api/models/tanh17042004/dta_legal_model_sup "HTTP/1.1 200 OK"


In [30]:
import logging
import random

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from datasets import load_dataset

logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)



# For this dataset, we want to concatenate the title and texts for the corpus
corpus = corpus.map(lambda x: {"text": (x["title"] + " " + x["text"]).strip()}, remove_columns=["title"])

# Shrink the corpus size heavily to only the relevant documents + 30,000 random documents
required_corpus_ids = set(map(str, relevant_docs_data["corpus_id"]))
required_corpus_ids |= set(random.sample(corpus["id"], k=30_000))
corpus = corpus.filter(lambda x: x["id"] in required_corpus_ids)

# Convert the datasets to dictionaries
corpus = dict(zip(corpus["id"], corpus["text"]))  # Our corpus (cid => document)
queries = dict(zip(queries["query_id"], queries["question"]))  # Our queries (qid => question)
relevant_docs = {}  # Query ID to relevant documents (qid => set([relevant_cids])
for qid, corpus_ids in zip(relevant_docs_data["query_id"], relevant_docs_data["corpus_id"]):
    qid = str(qid)
    corpus_ids = str(corpus_ids)
    if qid not in relevant_docs:
        relevant_docs[qid] = set()
    relevant_docs[qid].add(corpus_ids)

# Given queries, a corpus and a mapping with relevant documents, the InformationRetrievalEvaluator computes different IR metrics.
ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    mrr_at_k=[1, 3, 5, 10],
    ndcg_at_k=[1, 3, 5, 10],
    precision_recall_at_k=[1, 3, 5, 10],
    accuracy_at_k=[1,3,5,10],
    map_at_k=[10, 100],
    name="dta_legal_model",
)
results = ir_evaluator(legal_dta_model)

Filter:   0%|          | 0/61425 [00:00<?, ? examples/s]

2026-04-28 14:25:22 - Information Retrieval Evaluation of the model on the dta_legal_model dataset:
2026-04-28 14:30:13 - Queries: 2556
2026-04-28 14:30:13 - Corpus: 30917

2026-04-28 14:30:14 - Score-Function: cosine
2026-04-28 14:30:14 - Accuracy@1: 80.28%
2026-04-28 14:30:14 - Accuracy@3: 95.19%
2026-04-28 14:30:14 - Accuracy@5: 97.42%
2026-04-28 14:30:14 - Accuracy@10: 98.98%
2026-04-28 14:30:14 - Precision@1: 80.28%
2026-04-28 14:30:14 - Precision@3: 31.73%
2026-04-28 14:30:14 - Precision@5: 19.48%
2026-04-28 14:30:14 - Precision@10: 9.90%
2026-04-28 14:30:14 - Recall@1: 80.28%
2026-04-28 14:30:14 - Recall@3: 95.19%
2026-04-28 14:30:14 - Recall@5: 97.42%
2026-04-28 14:30:14 - Recall@10: 98.98%
2026-04-28 14:30:14 - MRR@1: 0.8028
2026-04-28 14:30:14 - MRR@3: 0.8712
2026-04-28 14:30:14 - MRR@5: 0.8764
2026-04-28 14:30:14 - MRR@10: 0.8785
2026-04-28 14:30:14 - NDCG@1: 0.8028
2026-04-28 14:30:14 - NDCG@3: 0.8920
2026-04-28 14:30:14 - NDCG@5: 0.9013
2026-04-28 14:30:14 - NDCG@10: 0.906

In [31]:
legal_huy_model = SentenceTransformer("huyydangg/DEk21_hcmute_embedding")


2026-04-28 14:32:58 - Use pytorch device_name: cuda:0
2026-04-28 14:32:58 - Load pretrained SentenceTransformer: huyydangg/DEk21_hcmute_embedding
2026-04-28 14:32:58 - HTTP Request: HEAD https://huggingface.co/huyydangg/DEk21_hcmute_embedding/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:32:58 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/huyydangg/DEk21_hcmute_embedding/501df2abd66bfecf9f294c4d17741b0d9f3ebb7e/modules.json "HTTP/1.1 200 OK"
2026-04-28 14:32:58 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/huyydangg/DEk21_hcmute_embedding/501df2abd66bfecf9f294c4d17741b0d9f3ebb7e/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

2026-04-28 14:32:58 - HTTP Request: HEAD https://huggingface.co/huyydangg/DEk21_hcmute_embedding/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:32:59 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/huyydangg/DEk21_hcmute_embedding/501df2abd66bfecf9f294c4d17741b0d9f3ebb7e/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-28 14:32:59 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/huyydangg/DEk21_hcmute_embedding/501df2abd66bfecf9f294c4d17741b0d9f3ebb7e/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

2026-04-28 14:32:59 - HTTP Request: HEAD https://huggingface.co/huyydangg/DEk21_hcmute_embedding/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:32:59 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/huyydangg/DEk21_hcmute_embedding/501df2abd66bfecf9f294c4d17741b0d9f3ebb7e/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-28 14:32:59 - HTTP Request: HEAD https://huggingface.co/huyydangg/DEk21_hcmute_embedding/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:32:59 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/huyydangg/DEk21_hcmute_embedding/501df2abd66bfecf9f294c4d17741b0d9f3ebb7e/README.md "HTTP/1.1 200 OK"
2026-04-28 14:32:59 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/huyydangg/DEk21_hcmute_embedding/501df2abd66bfecf9f294c4d17741b0d9f3ebb7e/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

... (16 outputs left out)

bpe.codes: 0.00B [00:00, ?B/s]

2026-04-28 14:33:09 - HTTP Request: HEAD https://huggingface.co/huyydangg/DEk21_hcmute_embedding/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 14:33:09 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/huyydangg/DEk21_hcmute_embedding/501df2abd66bfecf9f294c4d17741b0d9f3ebb7e/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-04-28 14:33:09 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/huyydangg/DEk21_hcmute_embedding/501df2abd66bfecf9f294c4d17741b0d9f3ebb7e/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

2026-04-28 14:33:09 - HTTP Request: GET https://huggingface.co/api/models/huyydangg/DEk21_hcmute_embedding "HTTP/1.1 200 OK"


In [33]:
import logging
import random

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from datasets import load_dataset

logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)



# For this dataset, we want to concatenate the title and texts for the corpus
corpus = corpus.map(lambda x: {"text": (x["title"] + " " + x["text"]).strip()}, remove_columns=["title"])

# Shrink the corpus size heavily to only the relevant documents + 30,000 random documents
required_corpus_ids = set(map(str, relevant_docs_data["corpus_id"]))
required_corpus_ids |= set(random.sample(corpus["id"], k=30_000))
corpus = corpus.filter(lambda x: x["id"] in required_corpus_ids)

# Convert the datasets to dictionaries
corpus = dict(zip(corpus["id"], corpus["text"]))  # Our corpus (cid => document)
queries = dict(zip(queries["query_id"], queries["question"]))  # Our queries (qid => question)
relevant_docs = {}  # Query ID to relevant documents (qid => set([relevant_cids])
for qid, corpus_ids in zip(relevant_docs_data["query_id"], relevant_docs_data["corpus_id"]):
    qid = str(qid)
    corpus_ids = str(corpus_ids)
    if qid not in relevant_docs:
        relevant_docs[qid] = set()
    relevant_docs[qid].add(corpus_ids)

# Given queries, a corpus and a mapping with relevant documents, the InformationRetrievalEvaluator computes different IR metrics.
ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    mrr_at_k=[1, 3, 5, 10],
    ndcg_at_k=[1, 3, 5, 10],
    precision_recall_at_k=[1, 3, 5, 10],
    accuracy_at_k=[1,3,5,10],
    map_at_k=[10, 100],
    name="huy_legal_model",
)
results = ir_evaluator(legal_huy_model)

Filter:   0%|          | 0/61425 [00:00<?, ? examples/s]

2026-04-28 14:33:51 - Information Retrieval Evaluation of the model on the huy_legal_model dataset:
2026-04-28 14:38:42 - Queries: 2556
2026-04-28 14:38:42 - Corpus: 30891

2026-04-28 14:38:42 - Score-Function: cosine
2026-04-28 14:38:42 - Accuracy@1: 79.97%
2026-04-28 14:38:42 - Accuracy@3: 93.82%
2026-04-28 14:38:42 - Accuracy@5: 96.28%
2026-04-28 14:38:42 - Accuracy@10: 98.63%
2026-04-28 14:38:42 - Precision@1: 79.97%
2026-04-28 14:38:42 - Precision@3: 31.27%
2026-04-28 14:38:42 - Precision@5: 19.26%
2026-04-28 14:38:42 - Precision@10: 9.86%
2026-04-28 14:38:42 - Recall@1: 79.97%
2026-04-28 14:38:42 - Recall@3: 93.82%
2026-04-28 14:38:42 - Recall@5: 96.28%
2026-04-28 14:38:42 - Recall@10: 98.63%
2026-04-28 14:38:42 - MRR@1: 0.7997
2026-04-28 14:38:42 - MRR@3: 0.8632
2026-04-28 14:38:42 - MRR@5: 0.8688
2026-04-28 14:38:42 - MRR@10: 0.8721
2026-04-28 14:38:42 - NDCG@1: 0.7997
2026-04-28 14:38:42 - NDCG@3: 0.8826
2026-04-28 14:38:42 - NDCG@5: 0.8927
2026-04-28 14:38:42 - NDCG@10: 0.900

In [5]:
legal_huy_model = SentenceTransformer("mainguyen9/vietlegal-harrier-0.6b")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.38G [00:00<?, ?B/s]

... (5 outputs left out)

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [6]:
import logging
import random

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from datasets import load_dataset

logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)



# For this dataset, we want to concatenate the title and texts for the corpus
corpus = corpus.map(lambda x: {"text": (x["title"] + " " + x["text"]).strip()}, remove_columns=["title"])

# Shrink the corpus size heavily to only the relevant documents + 30,000 random documents
required_corpus_ids = set(map(str, relevant_docs_data["corpus_id"]))
required_corpus_ids |= set(random.sample(corpus["id"], k=30_000))
corpus = corpus.filter(lambda x: x["id"] in required_corpus_ids)

# Convert the datasets to dictionaries
corpus = dict(zip(corpus["id"], corpus["text"]))  # Our corpus (cid => document)
queries = dict(zip(queries["query_id"], queries["question"]))  # Our queries (qid => question)
relevant_docs = {}  # Query ID to relevant documents (qid => set([relevant_cids])
for qid, corpus_ids in zip(relevant_docs_data["query_id"], relevant_docs_data["corpus_id"]):
    qid = str(qid)
    corpus_ids = str(corpus_ids)
    if qid not in relevant_docs:
        relevant_docs[qid] = set()
    relevant_docs[qid].add(corpus_ids)

# Given queries, a corpus and a mapping with relevant documents, the InformationRetrievalEvaluator computes different IR metrics.
ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    mrr_at_k=[1, 3, 5, 10],
    ndcg_at_k=[1, 3, 5, 10],
    precision_recall_at_k=[1, 3, 5, 10],
    accuracy_at_k=[1,3,5,10],
    map_at_k=[10, 100],
    name="huy_legal_model",
)
results = ir_evaluator(legal_huy_model)

Map:   0%|          | 0/61425 [00:00<?, ? examples/s]

Filter:   0%|          | 0/61425 [00:00<?, ? examples/s]

2026-04-29 08:00:42 - Information Retrieval Evaluation of the model on the huy_legal_model dataset:
2026-04-29 09:07:46 - Queries: 2556
2026-04-29 09:07:46 - Corpus: 30928

2026-04-29 09:07:46 - Score-Function: cosine
2026-04-29 09:07:46 - Accuracy@1: 53.79%
2026-04-29 09:07:46 - Accuracy@3: 74.73%
2026-04-29 09:07:46 - Accuracy@5: 80.44%
2026-04-29 09:07:46 - Accuracy@10: 86.97%
2026-04-29 09:07:46 - Precision@1: 53.79%
2026-04-29 09:07:46 - Precision@3: 24.91%
2026-04-29 09:07:46 - Precision@5: 16.09%
2026-04-29 09:07:46 - Precision@10: 8.70%
2026-04-29 09:07:46 - Recall@1: 53.79%
2026-04-29 09:07:46 - Recall@3: 74.73%
2026-04-29 09:07:46 - Recall@5: 80.44%
2026-04-29 09:07:46 - Recall@10: 86.97%
2026-04-29 09:07:46 - MRR@1: 0.5379
2026-04-29 09:07:46 - MRR@3: 0.6331
2026-04-29 09:07:46 - MRR@5: 0.6462
2026-04-29 09:07:46 - MRR@10: 0.6551
2026-04-29 09:07:46 - NDCG@1: 0.5379
2026-04-29 09:07:46 - NDCG@3: 0.6625
2026-04-29 09:07:46 - NDCG@5: 0.6861
2026-04-29 09:07:46 - NDCG@10: 0.707

In [9]:
legal_huy_model = SentenceTransformer("BAAI/bge-m3")


2026-04-30 15:18:17 - Use pytorch device_name: cuda:0
2026-04-30 15:18:17 - Load pretrained SentenceTransformer: BAAI/bge-m3
2026-04-30 15:18:17 - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-30 15:18:17 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
2026-04-30 15:18:17 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

2026-04-30 15:18:17 - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-30 15:18:17 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-30 15:18:17 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

2026-04-30 15:18:17 - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-30 15:18:17 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-30 15:18:18 - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-30 15:18:18 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/README.md "HTTP/1.1 200 OK"
2026-04-30 15:18:18 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

... (17 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

2026-04-30 15:18:27 - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-04-30 15:18:38 - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3 "HTTP/1.1 200 OK"
2026-04-30 15:18:38 - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-30 15:18:38 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-04-30 15:18:38 - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

2026-04-30 15:18:38 - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3 "HTTP/1.1 200 OK"


In [11]:
import logging
import random

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from datasets import load_dataset

logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)



# For this dataset, we want to concatenate the title and texts for the corpus
corpus = corpus.map(lambda x: {"text": (x["title"] + " " + x["text"]).strip()}, remove_columns=["title"])

# Shrink the corpus size heavily to only the relevant documents + 30,000 random documents
required_corpus_ids = set(map(str, relevant_docs_data["corpus_id"]))
required_corpus_ids |= set(random.sample(corpus["id"], k=30_000))
corpus = corpus.filter(lambda x: x["id"] in required_corpus_ids)

# Convert the datasets to dictionaries
corpus = dict(zip(corpus["id"], corpus["text"]))  # Our corpus (cid => document)
queries = dict(zip(queries["query_id"], queries["question"]))  # Our queries (qid => question)
relevant_docs = {}  # Query ID to relevant documents (qid => set([relevant_cids])
for qid, corpus_ids in zip(relevant_docs_data["query_id"], relevant_docs_data["corpus_id"]):
    qid = str(qid)
    corpus_ids = str(corpus_ids)
    if qid not in relevant_docs:
        relevant_docs[qid] = set()
    relevant_docs[qid].add(corpus_ids)

# Given queries, a corpus and a mapping with relevant documents, the InformationRetrievalEvaluator computes different IR metrics.
ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    mrr_at_k=[1, 3, 5, 10],
    ndcg_at_k=[1, 3, 5, 10],
    precision_recall_at_k=[1, 3, 5, 10],
    accuracy_at_k=[1,3,5,10],
    map_at_k=[10, 100],
    name="_legal_model",
)
results = ir_evaluator(legal_huy_model)

Filter:   0%|          | 0/61425 [00:00<?, ? examples/s]

2026-04-30 15:19:00 - Information Retrieval Evaluation of the model on the _legal_model dataset:


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 281.81 MiB is free. Including non-PyTorch memory, this process has 14.29 GiB memory in use. Of the allocated memory 8.27 GiB is allocated by PyTorch, and 5.89 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)